# Kaggle: LLM-as-judge scoring across all 5 runs

Second half of the project's dual-metric evaluation (deterministic rubric already done, `results/summary.csv`). Judges the *same* 150-id common subsample (`data/raw/llm_judge_ids.json`, seeded) across `baseline`, `sft_qlora`, `sft_lora_fp`, `dpo_from_sft`, `dpo_from_base` -- apples-to-apples, and affordable (750 judge calls total) rather than the full 803-row test set x 5 runs.

Same Ollama-on-Kaggle-T4 approach as `kaggle_teacher_gen.ipynb` (`src/eval/llm_judge.py` reuses that script's HTTP/concurrency pattern directly) -- validated locally first on 3 real rows (see `LOG.md` 2026-08-20) before this run: parser handled real judge output correctly, and the scores were genuinely differentiated with substantive rationale, not just rubber-stamped 5s.

**Before running:**
1. Zip `src/` (contents) + `config.yaml` as `src.zip`.
2. Upload that zip plus `results/{baseline,sft_qlora,sft_lora_fp,dpo_from_sft,dpo_from_base}_gen.jsonl` and `data/raw/llm_judge_ids.json` as a Kaggle Dataset.
3. GPU accelerator (T4 x1), Internet on (needed to install Ollama and pull the judge model).
4. Run all cells.

**Output:** `/kaggle/working/results/{run_id}_judged.jsonl` for each of the 5 runs, plus a printed summary table -- download and place in local `results/`.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q requests pyyaml

In [ ]:
import os, subprocess, time, requests

OLLAMA_NUM_PARALLEL = 8
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = str(OLLAMA_NUM_PARALLEL)

subprocess.Popen(["ollama", "serve"], env=env,
                  stdout=open("/kaggle/working/ollama_serve.log", "w"),
                  stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=2)
        break
    # Catches both "port not open yet" (ConnectionError) and "port open
    # but server not yet answering requests" (ReadTimeout/Timeout) -- a
    # real run hit the latter (Ollama's own install log had already
    # printed "API is now available" before the server was actually
    # ready to respond), which this loop didn't retry on originally
    # since it only caught ConnectionError. See LOG.md 2026-08-21.
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up -- check /kaggle/working/ollama_serve.log")
print("Ollama server is up.")

JUDGE_MODEL = "qwen2.5:7b-instruct-q4_0"
subprocess.run(["ollama", "pull", JUDGE_MODEL], check=True)

# Warmup generate so `ollama ps` reports something, and to catch a
# CPU-fallback here rather than partway through the real run.
requests.post("http://localhost:11434/api/generate",
               json={"model": JUDGE_MODEL, "prompt": "hello", "stream": False})
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print(ps.stdout)
if "100% GPU" not in ps.stdout and "GPU" not in ps.stdout.split("\n")[-2]:
    print("WARNING: model may not be fully on GPU -- check the PROCESSOR column above.")

In [ ]:
import sys, zipfile

def find_repo(root="/kaggle/input"):
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir} -- could not find src/data/build_irac.py under /kaggle/input "
        "(in any nesting). Check the dataset is attached."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)

from src.eval.llm_judge import run as judge_run

In [ ]:
def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

IDS_FILE = find_data_file("llm_judge_ids.json")
RUN_IDS = ["baseline", "sft_qlora", "sft_lora_fp", "dpo_from_sft", "dpo_from_base"]
GEN_FILES = {run_id: find_data_file(f"{run_id}_gen.jsonl") for run_id in RUN_IDS}
for run_id, path in GEN_FILES.items():
    print(run_id, "->", path)

RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Judge each run against the same 150-id subsample

Sequential, not parallel across runs (keeps `OLLAMA_NUM_PARALLEL` fully available to each run's own internal concurrency rather than splitting it across 5 simultaneous run-level submissions).

In [ ]:
for run_id in RUN_IDS:
    print(f"=== {run_id} ===")
    judge_run(
        input_path=GEN_FILES[run_id],
        output_path=os.path.join(RESULTS_DIR, f"{run_id}_judged.jsonl"),
        model="qwen2.5:7b-instruct-q4_0",
        host="http://localhost:11434",
        concurrency=OLLAMA_NUM_PARALLEL,
        ids_file=IDS_FILE,
    )

## Summary

In [ ]:
import json

print(f"{'run_id':<15} {'n':>4} {'groundedness':>13} {'reasoning':>10} {'overall':>8} {'parse_ok':>9}")
for run_id in RUN_IDS:
    path = os.path.join(RESULTS_DIR, f"{run_id}_judged.jsonl")
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    ok = [r for r in rows if r.get("judge_parse_ok")]
    n = len(rows)
    if ok:
        g = sum(r["groundedness"] for r in ok) / len(ok)
        rq = sum(r["reasoning_quality"] for r in ok) / len(ok)
        o = sum(r["overall"] for r in ok) / len(ok)
    else:
        g = rq = o = float("nan")
    print(f"{run_id:<15} {n:>4} {g:>13.2f} {rq:>10.2f} {o:>8.2f} {len(ok)/n:>9.1%}")